# Reward Weight Ablation
Measure how changing the style / meaning / fluency weights affects the total reward
and ranking of candidates on the test set. All weight triples are normalised to sum to 1.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from rl_rewriter.config import ModelConfig, ProjectConfig, RewardWeights
from rl_rewriter.dataset import clean_dataset, load_dataset
from rl_rewriter.scoring import RewardScorer

TEST_PATH = REPO_ROOT / 'data' / 'processed' / 'test.csv'
dataset = clean_dataset(load_dataset(TEST_PATH))
print(f'Test rows loaded: {len(dataset)}')

## Baseline scores with default weights (style=0.45, meaning=0.40, fluency=0.15)

In [ ]:
default_config = ProjectConfig()
scorer = RewardScorer(default_config.reward_weights, default_config.model)

results = []
for r in dataset:
    bd = scorer.calculate_reward(r['original_text'], r['styled_text'], r['target_style'])
    results.append({'style': r['target_style'], 'style_s': bd.style, 'meaning_s': bd.meaning,
                    'fluency_s': bd.fluency, 'total': bd.total})

n = len(results) or 1
print(f"Avg style:   {sum(r['style_s'] for r in results)/n:.3f}")
print(f"Avg meaning: {sum(r['meaning_s'] for r in results)/n:.3f}")
print(f"Avg fluency: {sum(r['fluency_s'] for r in results)/n:.3f}")
print(f"Avg total:   {sum(r['total'] for r in results)/n:.3f}")

## Ablation: vary weight configurations

In [ ]:
WEIGHT_CONFIGS = [
    ('default',           0.45, 0.40, 0.15),
    ('style-heavy',       0.70, 0.20, 0.10),
    ('meaning-heavy',     0.20, 0.70, 0.10),
    ('fluency-heavy',     0.20, 0.20, 0.60),
    ('equal',             0.33, 0.34, 0.33),
    ('style-only',        1.00, 0.00, 0.00),
    ('meaning-only',      0.00, 1.00, 0.00),
    ('fluency-only',      0.00, 0.00, 1.00),
]

summary_rows = []
model_config = ModelConfig()

for label, ws, wm, wf in WEIGHT_CONFIGS:
    weights = RewardWeights(style=ws, meaning=wm, fluency=wf)
    s = RewardScorer(weights, model_config)
    totals = [s.calculate_reward(r['original_text'], r['styled_text'], r['target_style']).total
              for r in dataset]
    avg = sum(totals) / max(len(totals), 1)
    summary_rows.append((label, ws, wm, wf, avg))

print(f"{'Config':<20} {'Style':>6} {'Meaning':>8} {'Fluency':>8} {'Avg Total':>10}")
print('-' * 56)
for label, ws, wm, wf, avg in summary_rows:
    print(f'{label:<20} {ws:>6.2f} {wm:>8.2f} {wf:>8.2f} {avg:>10.3f}')

## Per-style breakdown for each weight config

In [ ]:
from collections import defaultdict

for label, ws, wm, wf in WEIGHT_CONFIGS:
    weights = RewardWeights(style=ws, meaning=wm, fluency=wf)
    s = RewardScorer(weights, model_config)
    by_style = defaultdict(list)
    for r in dataset:
        total = s.calculate_reward(r['original_text'], r['styled_text'], r['target_style']).total
        by_style[r['target_style']].append(total)
    per_style = {k: sum(v)/len(v) for k, v in by_style.items()}
    print(f"{label}: {per_style}")

## Ranking stability: how often does the best candidate change across weight configs?

In [ ]:
# For each record, generate 5 dummy candidates using the fallback generator
# (no model download needed) and check if top-1 changes across weight configs.
from rl_rewriter.generator import TextGenerator

gen = TextGenerator(model_config)
disagreements = 0
total_checked = 0

for r in dataset[:20]:  # limit to 20 for speed
    candidates = gen.generate_candidates(r['original_text'], r['target_style'], num_candidates=5)
    rankings = {}
    for label, ws, wm, wf in WEIGHT_CONFIGS:
        weights = RewardWeights(style=ws, meaning=wm, fluency=wf)
        s = RewardScorer(weights, model_config)
        scored = [(c.text, s.calculate_reward(r['original_text'], c.text, r['target_style']).total)
                  for c in candidates]
        best = max(scored, key=lambda x: x[1])[0]
        rankings[label] = best
    unique_bests = len(set(rankings.values()))
    if unique_bests > 1:
        disagreements += 1
    total_checked += 1

print(f'Records where top-1 changed across weight configs: {disagreements}/{total_checked}')
print(f'Stability rate: {(1 - disagreements/max(total_checked,1))*100:.1f}%')